# LipVoicer Fine-tuning — Stage 2 + Stage 1

**Pipeline:**
1. Download LRS2 pretrained checkpoints
2. Preprocess: WAV→mel-spec, MP4→face frames
3. Fine-tune Stage 2 AudioVisualModel (freeze lipreader+face, train DiffWave)
4. Stage 1: pretrained AVSR zero-shot; fine-tune CTC if WER >60%
5. E2E: Stage1 → Stage2 → HiFi-GAN → WAV
6. Metrics: STOI, PESQ, WER; guidance ablation

**Hardware:** RTX 4090 (24 GB), bf16, batch=8

In [1]:
import os, sys, re, json, math, time, random, shutil, subprocess, contextlib
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

def find_pipe_root(start: Path) -> Path:
    for cand in [start.resolve(), *start.resolve().parents]:
        if (cand / 'pyproject.toml').exists() and (cand / 'third_party' / 'LipVoicer').exists():
            return cand
    raise FileNotFoundError('Could not locate Pipeline root from current working directory')

PIPE_ROOT = find_pipe_root(Path.cwd())
PROJ_ROOT = PIPE_ROOT.parent
LV_ROOT   = PIPE_ROOT / 'third_party' / 'LipVoicer'

data_root_candidates = [
    PIPE_ROOT / 'data' / 'custom_data',
    PROJ_ROOT / 'data' / 'custom_data',
]
DATA_ROOT = next(
    (cand for cand in data_root_candidates if (cand / 'dataset_final' / 'train.tsv').exists()),
    next((cand for cand in data_root_candidates if cand.exists()), data_root_candidates[0])
)
MANIFEST_DIR = DATA_ROOT / 'dataset_final'
AUDIO_DIR    = MANIFEST_DIR / 'audios'
VIDEO_DIR    = MANIFEST_DIR / 'videos'
SEGMENTS_DIR = DATA_ROOT / 'segments'
ROI_DIR      = DATA_ROOT / 'lip_rois'
FACE_DIR     = DATA_ROOT / 'faces'
MEL_DIR      = DATA_ROOT / 'mel_specs'
OUTPUT_DIR   = PIPE_ROOT / 'outputs' / 'stage2_finetune'
STAGE1_OUT   = PIPE_ROOT / 'outputs' / 'stage1_eval'

def clip_audio_path(clip_id: str, speaker_id: str) -> Path:
    candidates = [
        AUDIO_DIR / f'{clip_id}.wav',
        SEGMENTS_DIR / speaker_id / f'{clip_id}.wav',
    ]
    return next((p for p in candidates if p.exists()), candidates[-1])

def clip_video_path(clip_id: str, speaker_id: str) -> Path:
    candidates = [
        VIDEO_DIR / f'{clip_id}.mp4',
        SEGMENTS_DIR / speaker_id / f'{clip_id}.mp4',
    ]
    return next((p for p in candidates if p.exists()), candidates[-1])

for d in [FACE_DIR, MEL_DIR, OUTPUT_DIR, STAGE1_OUT]:
    d.mkdir(parents=True, exist_ok=True)

for p in [str(PROJ_ROOT), str(PIPE_ROOT), str(LV_ROOT)]:
    if p not in sys.path:
        sys.path.insert(0, p)

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PIPE_ROOT:', PIPE_ROOT)
print('DATA_ROOT:', DATA_ROOT)
print('Device :', DEVICE)
print('GPU    :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A')
print('Torch  :', torch.__version__)

CFG = {
    'sr':16000,'filter_length':640,'hop_length':160,'win_length':640,
    'mel_fmin':20.0,'mel_fmax':8000.0,'n_mels':80,
    'fps':25,'video_window':25,
    's2_lr':2e-5,'s2_batch':8,'s2_iters':5000,
    's2_ckpt_every':500,'s2_log_every':50,
    's2_cond_drop_prob':0.2,'s2_grad_clip':1.0,
    'T':400,'beta_0':0.0001,'beta_T':0.02,
    's1_lr':2e-4,'s1_epochs':200,'s1_batch':8,
    's1_warmup':500,'s1_d_model':256,'s1_nhead':8,'s1_layers':4,
    'w_video':2.0,'w_asr':1.5,'num_workers':4,
}
CFG['vid_2_aud'] = CFG['sr'] / CFG['fps'] / CFG['hop_length']  # 4.0
CFG['mel_window'] = int(CFG['video_window'] * CFG['vid_2_aud']) # 100
print(f"mel frames/video frame: {CFG['vid_2_aud']}   mel window: {CFG['mel_window']}")

/home/shra012/Workspace/data-255/project/Pipeline/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PIPE_ROOT: /home/shra012/Workspace/data-255/project/Pipeline
DATA_ROOT: /home/shra012/Workspace/data-255/project/Pipeline/data/custom_data
Device : cuda
GPU    : NVIDIA GeForce RTX 4090
Torch  : 2.11.0+cu130
mel frames/video frame: 4.0   mel window: 100


## 1. Checkpoint Downloads

In [2]:
import gdown

CKPTS = {
    'MelGen_LRS2':   {'id':'1gTIpxaMx31ZUhPd2jW8Bt8u4QknieO31',
                      'rel':'exp/LRS2/wnet_h512_d12_T400_betaT0.02/checkpoint/1000000.pkl'},
    'HiFiGAN':       {'id':'1h0gcgifwe5HVM76rlREHj1daBNItWh7e',
                      'rel':'hifi_gan/g_02400000'},
    'LipRead_LRS3':  {'id':'1t8RHhzDTTvOQkLQhmK1LZGnXRRXOXGi6',
                      'rel':'mouthroi_processing/benchmarks/LRS3/models/LRS3_V_WER19.1.zip',
                      'unzip':True},
    'ASR_LRS2':      {'id':'1adeCf4NzhshJVU-JndlKpC34rRwJOQ2B',
                      'rel':'ASR/callbacks/LRS23/AO/EffConfCTC/checkpoints_ft_lrs2.ckpt'},
    'Tokenizer':     {'id':'1u3U3aHaTWvR_NTftkUGv1JXkxpX1pkOL',
                      'rel':'ASR/media/tokenizerbpe256.model'},
    'LM':            {'id':'1PSo4ZQIZPWEI_S5LHkJBo0gYhQpWzRnh',
                      'rel':'ASR/callbacks/LRS23/LM/GPT-Small/checkpoints_epoch_10_step_2860.ckpt'},
}

def dl(name, spec, root=LV_ROOT):
    dest = root / spec['rel']
    if dest.exists():
        print(f'  [{name}] exists ({dest.stat().st_size/1e6:.0f} MB)')
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    gdown.download(f"https://drive.google.com/uc?id={spec['id']}", str(dest), quiet=False)
    if spec.get('unzip') and dest.exists():
        import zipfile
        with zipfile.ZipFile(dest) as z:
            z.extractall(str(dest.parent))
        print(f'  [{name}] unzipped')
    return dest

ckpt_paths = {k: dl(k, v) for k, v in CKPTS.items()}
MELGEN_CKPT  = ckpt_paths['MelGen_LRS2']
HIFIGAN_CKPT = ckpt_paths['HiFiGAN']
print('\nAll downloads done:', all(p.exists() for p in ckpt_paths.values()))

  [MelGen_LRS2] exists (685 MB)
  [HiFiGAN] exists (56 MB)
  [LipRead_LRS3] exists (937 MB)
  [ASR_LRS2] exists (411 MB)
  [Tokenizer] exists (0 MB)
  [LM] exists (347 MB)

All downloads done: True


## 2. Data Audit

In [3]:
manifest_paths = {
    'train': MANIFEST_DIR/'train.tsv',
    'val': MANIFEST_DIR/'val.tsv',
    'test': MANIFEST_DIR/'test.tsv',
}
print('Loading manifests from:', MANIFEST_DIR)
missing = [name for name, path in manifest_paths.items() if not path.exists()]
if missing:
    raise FileNotFoundError(
        f'Missing manifests {missing} under {MANIFEST_DIR}. '
        f'DATA_ROOT={DATA_ROOT} PIPE_ROOT={PIPE_ROOT}'
    )

train_df = pd.read_csv(manifest_paths['train'], sep='\t')
val_df   = pd.read_csv(manifest_paths['val'],   sep='\t')
test_df  = pd.read_csv(manifest_paths['test'],  sep='\t')
all_df   = pd.concat([train_df,val_df,test_df], ignore_index=True)
print(f'train={len(train_df)}  val={len(val_df)}  test={len(test_df)}  total={len(all_df)}')

rows=[]
for _,r in all_df.iterrows():
    c,s=r['clip_id'],r['speaker_id']
    rows.append({'clip_id':c,'speaker_id':s,
                 'roi':(ROI_DIR/s/f'{c}.npz').exists(),
                 'wav':clip_audio_path(c, s).exists(),
                 'mp4':clip_video_path(c, s).exists()})
aud=pd.DataFrame(rows)
ok=aud[['roi','wav','mp4']].all(axis=1)
print(f'Complete: {ok.sum()}/{len(aud)}')
if (~ok).any(): display(aud[~ok])

t_vals=[np.load(ROI_DIR/r['speaker_id']/f"{r['clip_id']}.npz")['mouth_rois'].shape[0]
        for _,r in all_df.iterrows()]
print(f'ROI frames: min={min(t_vals)} max={max(t_vals)} mean={np.mean(t_vals):.1f}')
sample_cid = all_df.iloc[0]['clip_id']

Loading manifests from: /home/shra012/Workspace/data-255/project/Pipeline/data/custom_data/dataset_final
train=101  val=75  test=108  total=284
Complete: 0/284


,clip_id,speaker_id,roi,wav,mp4
0,spk_002_0001,spk_002,True,False,False
1,spk_002_0002,spk_002,True,False,False
2,spk_002_0003,spk_002,True,False,False
3,spk_002_0004,spk_002,True,False,False
4,spk_002_0005,spk_002,True,False,False
...,...,...,...,...,...
279,spk_003_0115,spk_003,True,False,False
280,spk_003_0116,spk_003,True,False,False
281,spk_003_0117,spk_003,True,False,False
282,spk_003_0118,spk_003,True,False,False


ROI frames: min=50 max=246 mean=120.6


## 3. Mel-spectrogram Generation

WAV → TacotronSTFT → save `{clip_id}.wav.spec`

In [6]:
import torchaudio
from Pipeline.third_party.LipVoicer.dataloaders.stft import TacotronSTFT, normalise_mel, denormalise_mel

stft_fn = TacotronSTFT(
    filter_length=CFG['filter_length'], hop_length=CFG['hop_length'],
    win_length=CFG['win_length'], sampling_rate=CFG['sr'],
    mel_fmin=CFG['mel_fmin'], mel_fmax=CFG['mel_fmax']
).to(DEVICE)

@torch.no_grad()
def wav_to_mel(wav_path):
    audio, sr = torchaudio.load(str(wav_path))
    if sr != CFG['sr']:
        audio = torchaudio.functional.resample(audio, sr, CFG['sr'])
    audio = audio.mean(0)
    audio = audio / (1.1 * audio.abs().max().clamp(min=1e-8))
    mel = stft_fn.mel_spectrogram(audio.unsqueeze(0).to(DEVICE))
    return mel.squeeze(0).cpu()

MEL_DIR.mkdir(parents=True, exist_ok=True)
skip=gen=fail=0
failed_clips = []
for _,r in tqdm(all_df.iterrows(), total=len(all_df), desc='mel-spec'):
    sp = MEL_DIR/f"{r['clip_id']}.wav.spec"
    if sp.exists(): skip+=1; continue
    wp = clip_audio_path(r['clip_id'], r['speaker_id'])
    if not wp.exists(): fail+=1; continue
    try: torch.save(wav_to_mel(wp), sp); gen+=1
    except Exception as e:
        print(f"WARN {r['clip_id']}: {e}")
        failed_clips.append((r['clip_id'], str(e)))
        fail+=1

print(f'Generated:{gen}  Skipped:{skip}  Failed:{fail}')
sample_spec = MEL_DIR/f'{sample_cid}.wav.spec'
if not sample_spec.exists():
    existing_specs = sorted(MEL_DIR.glob('*.wav.spec'))
    if not existing_specs:
        raise FileNotFoundError(
            f'No mel specs were generated in {MEL_DIR}. '
            f'First failures: {failed_clips[:5]}'
        )
    sample_spec = existing_specs[0]
    print(f'Sample clip {sample_cid} missing mel spec; using {sample_spec.name} for sanity check')
s=torch.load(sample_spec)
print(f'Sample mel shape: {tuple(s.shape)}   norm range: {normalise_mel(s).min():.2f}~{normalise_mel(s).max():.2f}')

mel-spec:   0%|          | 0/284 [00:00<?, ?it/s]


NameError: name 'clip_audio_path' is not defined

## 4. Face Frame Extraction

In [ ]:
import cv2
from PIL import Image

def extract_face(mp4_path, out_path):
    cap=cv2.VideoCapture(str(mp4_path))
    n=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, n//2)
    ok,frame=cap.read(); cap.release()
    if not ok: return False
    Image.fromarray(cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)).save(str(out_path))
    return True

skip=gen=fail=0
for _,r in tqdm(all_df.iterrows(), total=len(all_df), desc='face-frames'):
    op=FACE_DIR/f"{r['clip_id']}_face.jpg"
    if op.exists(): skip+=1; continue
    vp=clip_video_path(r['clip_id'], r['speaker_id'])
    if not vp.exists(): fail+=1; continue
    if extract_face(vp,op): gen+=1
    else: fail+=1
print(f'Generated:{gen}  Skipped:{skip}  Failed:{fail}')

## 5. Custom Dataset

Batch shapes: mel `(B,80,100)`, roi `(B,1,25,88,88)`, face `(B,3,224,224)`

In [ ]:
import torchvision.transforms as Tv
from Pipeline.third_party.LipVoicer.dataloaders.lipreading_utils import (
    Compose, Normalize as LVNorm, CenterCrop, RandomCrop, HorizontalFlip
)

def roi_tfm(split):
    c=(88,88); m,s=0.421,0.165
    base=[LVNorm(0.0,255.0)]
    aug=[RandomCrop(c),HorizontalFlip(0.5)] if split=='train' else [CenterCrop(c)]
    return Compose(base+aug+[LVNorm(m,s)])

def face_tfm():
    return Tv.Compose([Tv.Resize(224),Tv.CenterCrop(224),Tv.ToTensor(),
                       Tv.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

class LVDataset(Dataset):
    def __init__(self, manifest, split):
        self.df      = pd.read_csv(manifest,sep='\t').reset_index(drop=True)
        self.split   = split
        self.train   = split=='train'
        self.win     = CFG['video_window']
        self.mwin    = CFG['mel_window']
        self.v2a     = CFG['vid_2_aud']
        self.rtfm    = roi_tfm(split)
        self.ftfm    = face_tfm()

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        r   = self.df.iloc[idx]
        cid = r['clip_id']; spk = r['speaker_id']

        roi = np.load(ROI_DIR/spk/f'{cid}.npz')['mouth_rois']       # (T,96,96)
        mel = torch.load(MEL_DIR/f'{cid}.wav.spec')                  # (80,L)
        face= Image.open(FACE_DIR/f'{cid}_face.jpg').convert('RGB')

        if self.train:
            if roi.shape[0]<self.win or mel.shape[1]<self.mwin:
                return self.__getitem__(random.randrange(len(self)))
            sv  = random.randint(0, roi.shape[0]-self.win)
            roi = roi[sv:sv+self.win]
            sm  = int(sv*self.v2a)
            mel = mel[:,sm:sm+self.mwin]
            if mel.shape[1]<self.mwin:
                mel=F.pad(mel,(0,self.mwin-mel.shape[1]))
            if random.random()<0.5:
                from PIL.ImageEnhance import Brightness,Color
                face=Brightness(face).enhance(random.uniform(0.7,1.3))
                face=Color(face).enhance(random.uniform(0.7,1.3))

        roi_t  = torch.FloatTensor(self.rtfm(roi)).unsqueeze(0)  # (1,T,88,88)
        face_t = self.ftfm(face)                                  # (3,224,224)
        mel_n  = normalise_mel(mel)                               # (80,L)
        return mel_n, roi_t, face_t, cid

def collate(batch):
    m,r,f,c=zip(*batch)
    return torch.stack(m),torch.stack(r),torch.stack(f),list(c)

train_ds = LVDataset(MANIFEST_DIR/'train.tsv','train')
val_ds   = LVDataset(MANIFEST_DIR/'val.tsv','val')
test_ds  = LVDataset(MANIFEST_DIR/'test.tsv','test')

train_loader = DataLoader(train_ds,batch_size=CFG['s2_batch'],shuffle=True,
                          num_workers=CFG['num_workers'],collate_fn=collate,
                          pin_memory=True,drop_last=True)
val_loader   = DataLoader(val_ds,batch_size=4,shuffle=False,
                          num_workers=2,collate_fn=collate)

mb,rb,fb,cb=next(iter(train_loader))
print('mel :', tuple(mb.shape), '→ (8,80,100)')
print('roi :', tuple(rb.shape), '→ (8,1,25,88,88)')
print('face:', tuple(fb.shape), '→ (8,3,224,224)')

## 6. Stage 2 Model — Build + Load LRS2 Pretrained

Freeze `net_lipreading` + `net_facial`. Fine-tune `net_diffwave` only.

In [ ]:
from Pipeline.third_party.LipVoicer.models.audiovisual_model import AudioVisualModel
from Pipeline.third_party.LipVoicer.models.model_builder import ModelBuilder
from Pipeline.third_party.LipVoicer.utils import calc_diffusion_hyperparams, print_size

@contextlib.contextmanager
def cwd(path):
    old=os.getcwd(); os.chdir(path)
    try: yield
    finally: os.chdir(old)

# model_builder uses relative config path → must run from LV_ROOT
with cwd(LV_ROOT):
    builder = ModelBuilder()
    net_lip  = builder.build_lipreadingnet()
    net_face = builder.build_facial(fc_out=128, with_fc=True)
    model_cfg = dict(
        _name_='melgen', in_channels=80, out_channels=80,
        diffusion_step_embed_dim_in=128, diffusion_step_embed_dim_mid=512,
        diffusion_step_embed_dim_out=512, res_channels=512, skip_channels=512,
        num_res_layers=12, dilation_cycle=1, mel_upsample=[2,2]
    )
    net_dw   = builder.build_diffwave_model(model_cfg)

net = AudioVisualModel((net_lip, net_face, net_dw)).to(DEVICE)

# ── Load LRS2 pretrained checkpoint ──────────────────────────────────────────
def load_lv_checkpoint(model, ckpt_path):
    payload = torch.load(ckpt_path, map_location='cpu')
    sd = payload.get('model_state_dict', payload)
    # strip DataParallel wrapper keys
    sd = {k.replace('module.','',1): v for k,v in sd.items()}
    missing, unexpected = model.load_state_dict(sd, strict=False)
    print(f'  Missing : {len(missing)}')
    print(f'  Unexpected: {len(unexpected)}')
    if missing[:5]: print('  Missing sample:', missing[:5])
    return len(missing), len(unexpected)

print('Loading LRS2 MelGen checkpoint...')
m, u = load_lv_checkpoint(net, MELGEN_CKPT)
assert m == 0, f'{m} missing keys — checkpoint may not match architecture'
print('Checkpoint loaded cleanly.')

# ── Freeze lipreading + face encoder ─────────────────────────────────────────
for param in net.net_lipreading.parameters(): param.requires_grad = False
for param in net.net_facial.parameters():     param.requires_grad = False

trainable = sum(p.numel() for p in net.parameters() if p.requires_grad)
total     = sum(p.numel() for p in net.parameters())
print(f'Trainable: {trainable/1e6:.2f}M / {total/1e6:.2f}M total')
print('(DiffWave only — lipreader+face frozen)')

# ── Diffusion hyperparams ─────────────────────────────────────────────────────
diff_hp = calc_diffusion_hyperparams(T=CFG['T'], beta_0=CFG['beta_0'], beta_T=CFG['beta_T'])
diff_hp['Sigma'] = diff_hp['Sigma'].to(DEVICE)  # Sigma not moved by default

## 7. Stage 2 Fine-tuning

DDPM L1 noise-prediction loss. bf16 autocast. AdamW on DiffWave. 5000 iters.

In [ ]:
from torch.utils.tensorboard import SummaryWriter
from Pipeline.third_party.LipVoicer.dataloaders.stft import normalise_mel

criterion = nn.L1Loss()
optimizer = torch.optim.AdamW(
    [p for p in net.parameters() if p.requires_grad],
    lr=CFG['s2_lr'], weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG['s2_iters'], eta_min=CFG['s2_lr']*0.1
)

def training_loss(net, mel, roi, face, dh):
    """DDPM noise prediction loss (L1)."""
    T_steps = dh['T']; Alpha_bar = dh['Alpha_bar']
    B = mel.shape[0]
    t = torch.randint(T_steps, size=(B,1,1), device=DEVICE)
    z = torch.randn_like(mel)
    ab = Alpha_bar[t]                         # (B,1,1)
    x_t = ab.sqrt()*mel + (1-ab).sqrt()*z     # forward diffusion
    eps_hat = net(x_t, roi, face, t.view(B,1), CFG['s2_cond_drop_prob'])
    return criterion(eps_hat, z)

writer = SummaryWriter(log_dir=str(OUTPUT_DIR/'tb_logs'))

net.train()
net.net_lipreading.eval()  # BN layers in frozen modules stay in eval
net.net_facial.eval()

loader_iter = iter(train_loader)
best_val_loss = float('inf')
losses = []

print(f'Starting Stage 2 fine-tune: {CFG["s2_iters"]} iters, lr={CFG["s2_lr"]}')
t0 = time.time()

for step in range(1, CFG['s2_iters']+1):
    try:
        mel_b, roi_b, face_b, _ = next(loader_iter)
    except StopIteration:
        loader_iter = iter(train_loader)
        mel_b, roi_b, face_b, _ = next(loader_iter)

    mel_b  = mel_b.to(DEVICE)
    roi_b  = roi_b.to(DEVICE)
    face_b = face_b.to(DEVICE)

    optimizer.zero_grad(set_to_none=True)
    with torch.autocast('cuda', dtype=torch.bfloat16):
        loss = training_loss(net, mel_b, roi_b, face_b, diff_hp)

    loss.backward()
    torch.nn.utils.clip_grad_norm_(
        [p for p in net.parameters() if p.requires_grad], CFG['s2_grad_clip']
    )
    optimizer.step()
    scheduler.step()
    losses.append(float(loss))

    if step % CFG['s2_log_every'] == 0:
        avg = np.mean(losses[-CFG['s2_log_every']:])
        elapsed = time.time()-t0
        print(f'  step {step:5d}/{CFG["s2_iters"]}  loss={avg:.4f}  '
              f'lr={scheduler.get_last_lr()[0]:.2e}  elapsed={elapsed:.0f}s')
        writer.add_scalar('train/loss', avg, step)
        writer.add_scalar('train/lr', scheduler.get_last_lr()[0], step)

    if step % CFG['s2_ckpt_every'] == 0:
        # Validation loss
        net.eval()
        val_losses=[]
        with torch.no_grad():
            for vm,vr,vf,_ in val_loader:
                with torch.autocast('cuda',dtype=torch.bfloat16):
                    vl=training_loss(net,vm.to(DEVICE),vr.to(DEVICE),vf.to(DEVICE),diff_hp)
                val_losses.append(float(vl))
        val_loss = np.mean(val_losses)
        writer.add_scalar('val/loss', val_loss, step)
        print(f'  [ckpt {step}] val_loss={val_loss:.4f}')

        # Save checkpoint
        ckpt_path = OUTPUT_DIR/f'stage2_ft_{step:06d}.pkl'
        torch.save({'step':step,'model_state_dict':net.state_dict(),
                    'optimizer_state_dict':optimizer.state_dict(),
                    'val_loss':val_loss}, ckpt_path)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            shutil.copy(ckpt_path, OUTPUT_DIR/'stage2_ft_best.pkl')
            print(f'  → new best val_loss={best_val_loss:.4f}')

        net.train()
        net.net_lipreading.eval()
        net.net_facial.eval()

writer.close()
print(f'Done. Best val loss: {best_val_loss:.4f}')

In [ ]:
# Training curve plot
window=50
smooth=[np.mean(losses[max(0,i-window):i+1]) for i in range(len(losses))]
plt.figure(figsize=(10,4))
plt.plot(losses, alpha=0.3, label='raw')
plt.plot(smooth, label=f'smooth(w={window})')
plt.xlabel('iteration'); plt.ylabel('L1 loss')
plt.title('Stage 2 Fine-tuning Loss')
plt.legend(); plt.grid(True); plt.tight_layout()
plt.savefig(OUTPUT_DIR/'stage2_train_loss.png', dpi=120)
plt.show()

## 8. Stage 1 — Pretrained AVSR Zero-shot Evaluation

Runs `stage1_pretrained_eval.py` (requires lip-reading checkpoint downloaded in Section 1).

In [ ]:
import jiwer

s1_pretrained_out = STAGE1_OUT / 'pretrained'
s1_pretrained_out.mkdir(parents=True, exist_ok=True)

script = PIPE_ROOT / 'scripts' / 'stage1_pretrained_eval.py'
lip_roi_root = ROI_DIR

# Run pretrained eval on val split
cmd = [
    sys.executable, str(script),
    '--split', 'val',
    '--manifest-tsv', str(MANIFEST_DIR/'val.tsv'),
    '--lip-roi-root', str(lip_roi_root),
    '--output-dir', str(s1_pretrained_out),
]

print('Running Stage 1 pretrained eval...')
result = subprocess.run(cmd, capture_output=True, text=True,
                        cwd=str(PIPE_ROOT))
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])  # last 2000 chars
    print('\nNote: pretrained eval requires benchmark assets from Section 1.')
    print('If assets missing, WER defaults to 100% and we proceed with Stage 1 fine-tune.')
    S1_PRETRAINED_WER = 1.0
else:
    summary_path = s1_pretrained_out / 'val_summary_stage1_pretrained.json'
    if summary_path.exists():
        summ = json.loads(summary_path.read_text())
        S1_PRETRAINED_WER = summ['overall_wer']
        S1_PRETRAINED_CER = summ['overall_cer']
        print(f'Stage 1 pretrained WER: {S1_PRETRAINED_WER:.4f}')
        print(f'Stage 1 pretrained CER: {S1_PRETRAINED_CER:.4f}')
        print(f'Decision: {summ["decision"]}')
    else:
        S1_PRETRAINED_WER = 1.0

NEED_S1_FINETUNE = S1_PRETRAINED_WER > 0.60
print(f'\nNeed Stage 1 fine-tune: {NEED_S1_FINETUNE}  (WER={S1_PRETRAINED_WER:.3f})')

## 9. Stage 1 Fine-tuning (CTC lip-reader)

Trains `Stage1LipVoicerCNNCTC` (3D CNN + Transformer + CTC) on 101 custom clips.
Skipped if pretrained WER ≤ 0.60 from Section 8.

In [ ]:
# ── Vocab from train+val text ──────────────────────────────────────────────
all_text = pd.concat([train_df['text'],val_df['text']]).str.lower().tolist()
charset = sorted({ch for t in all_text for ch in t})
idx2tok = ['<blank>'] + charset
tok2idx = {t:i for i,t in enumerate(idx2tok)}
BLANK   = 0
VOCAB   = len(idx2tok)
print(f'Vocab: {VOCAB} (incl blank)  chars: {charset[:20]}...')

def encode(s): return [tok2idx[c] for c in s.lower().strip() if c in tok2idx]
def decode(ids):
    out=[]; prev=None
    for i in ids:
        if i!=BLANK and i!=prev: out.append(idx2tok[i])
        prev=i
    return ''.join(out)

In [ ]:
# ── Stage 1 model ─────────────────────────────────────────────────────────
class Stage1CTC(nn.Module):
    def __init__(self, vocab, d=256, heads=8, layers=4):
        super().__init__()
        self.frontend = nn.Sequential(
            nn.Conv3d(1,64,(5,7,7),(1,2,2),(2,3,3),bias=False),
            nn.BatchNorm3d(64), nn.ReLU(True),
            nn.MaxPool3d((1,3,3),(1,2,2),(0,1,1)),
        )
        self.frame_enc = nn.Sequential(
            nn.Conv2d(64,128,3,2,1), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.Conv2d(128,256,3,2,1), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.AdaptiveAvgPool2d((1,1)),
        )
        self.proj = nn.Linear(256,d)
        enc_layer = nn.TransformerEncoderLayer(
            d,heads,4*d,0.1,batch_first=True,activation='gelu'
        )
        self.encoder = nn.TransformerEncoder(enc_layer, layers)
        self.head    = nn.Linear(d, vocab)

    def forward(self, rois, lens, return_feats=False):
        # rois: (B,1,T,96,96)
        x = self.frontend(rois)               # (B,64,T,H,W)
        b,c,t,h,w = x.shape
        x = x.permute(0,2,1,3,4).reshape(b*t,c,h,w)
        x = self.frame_enc(x).view(b,t,-1)    # (B,T,256)
        x = self.proj(x)                       # (B,T,d)
        mask = torch.arange(t,device=x.device).unsqueeze(0) >= lens.unsqueeze(1)
        x = self.encoder(x, src_key_padding_mask=mask)
        logits = self.head(x)                  # (B,T,vocab)
        return (logits, x) if return_feats else logits

s1_model = Stage1CTC(VOCAB, CFG['s1_d_model'], CFG['s1_nhead'], CFG['s1_layers']).to(DEVICE)
print(f'Stage1 params: {sum(p.numel() for p in s1_model.parameters())/1e6:.2f}M')

In [ ]:
# ── Stage 1 Dataset ────────────────────────────────────────────────────────
class S1Dataset(Dataset):
    def __init__(self, df, split):
        self.df    = df.reset_index(drop=True)
        self.train = split=='train'

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        r   = self.df.iloc[idx]
        cid = r['clip_id']; spk = r['speaker_id']
        roi = np.load(ROI_DIR/spk/f'{cid}.npz')['mouth_rois'].astype(np.float32)/255.0
        # (T,96,96) → (1,T,96,96)
        if self.train:
            if random.random()<0.5:  # horizontal flip
                roi = roi[:,:,::-1].copy()
            # time mask: zero out up to 15% of frames
            T=roi.shape[0]; mask_len=int(T*0.15)
            if mask_len>0:
                st=random.randint(0,T-mask_len)
                roi[st:st+mask_len]=0.0
        roi_t  = torch.from_numpy(roi).unsqueeze(0)     # (1,T,96,96)
        toks_t = torch.tensor(encode(str(r['text'])), dtype=torch.long)
        return roi_t, roi_t.shape[1], toks_t, len(toks_t)

def s1_collate(batch):
    rois,rlens,toks,tlens=zip(*batch)
    B=len(rois); maxT=max(rlens); maxU=max(tlens)
    roi_pad=torch.zeros(B,1,maxT,96,96)
    tok_pad=torch.zeros(B,maxU,dtype=torch.long)
    for i,(r,rl,t,tl) in enumerate(zip(rois,rlens,toks,tlens)):
        roi_pad[i,:,:rl]=r; tok_pad[i,:tl]=t
    return roi_pad,torch.tensor(rlens),tok_pad,torch.tensor(tlens)

s1_train = DataLoader(S1Dataset(train_df,'train'),CFG['s1_batch'],shuffle=True,
                      num_workers=2,collate_fn=s1_collate,drop_last=False)
s1_val   = DataLoader(S1Dataset(val_df,'val'),4,shuffle=False,
                      num_workers=2,collate_fn=s1_collate)

In [ ]:
if not NEED_S1_FINETUNE:
    print('Stage 1 pretrained WER good enough, skipping fine-tune.')
    S1_BEST_CKPT = None
else:
    ctc_loss = nn.CTCLoss(blank=BLANK, zero_infinity=True)
    s1_opt   = torch.optim.AdamW(s1_model.parameters(), lr=CFG['s1_lr'], weight_decay=1e-4)

    total_steps = CFG['s1_epochs']*len(s1_train)
    def lr_lambda(step):
        if step < CFG['s1_warmup']: return step/max(1,CFG['s1_warmup'])
        progress=(step-CFG['s1_warmup'])/max(1,total_steps-CFG['s1_warmup'])
        return max(0.1, 0.5*(1+math.cos(math.pi*progress)))
    s1_sched = torch.optim.lr_scheduler.LambdaLR(s1_opt, lr_lambda)

    best_wer=1.0; global_step=0; S1_BEST_CKPT=None
    s1_train_losses=[]; s1_val_wers=[]

    for epoch in range(1, CFG['s1_epochs']+1):
        s1_model.train()
        ep_losses=[]
        for roi_b,rlen_b,tok_b,tlen_b in s1_train:
            roi_b=roi_b.to(DEVICE); rlen_b=rlen_b.to(DEVICE)
            tok_b=tok_b.to(DEVICE); tlen_b=tlen_b.to(DEVICE)

            with torch.autocast('cuda',dtype=torch.bfloat16):
                logits=s1_model(roi_b,rlen_b)              # (B,T,V)
                log_p=F.log_softmax(logits,dim=-1).permute(1,0,2)  # (T,B,V)
                tgt_flat=torch.cat([tok_b[i,:tlen_b[i]] for i in range(len(tlen_b))])
                loss=ctc_loss(log_p,tgt_flat,rlen_b,tlen_b)

            s1_opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(s1_model.parameters(),5.0)
            s1_opt.step(); s1_sched.step()
            ep_losses.append(float(loss)); global_step+=1

        # Validation WER every 10 epochs
        if epoch%10==0 or epoch==CFG['s1_epochs']:
            s1_model.eval()
            refs=[]; hyps=[]
            with torch.no_grad():
                for roi_b,rlen_b,tok_b,tlen_b in s1_val:
                    logits=s1_model(roi_b.to(DEVICE),rlen_b.to(DEVICE))
                    pred=logits.argmax(-1).cpu().numpy()
                    for j,(seq,tl) in enumerate(zip(pred,tlen_b)):
                        # greedy CTC decode
                        out=[]; prev=None
                        for tid in seq:
                            if tid!=BLANK and tid!=prev: out.append(tid)
                            prev=tid
                        hyps.append(decode(out))
                        refs.append(decode(tok_b[j,:tl].tolist()))
            val_wer=jiwer.wer(refs,hyps)
            val_cer=jiwer.cer(refs,hyps)
            s1_val_wers.append(val_wer)
            avg_loss=np.mean(ep_losses)
            print(f'  epoch {epoch:3d}/{CFG["s1_epochs"]}  loss={avg_loss:.4f}  '
                  f'WER={val_wer:.4f}  CER={val_cer:.4f}')

            if val_wer < best_wer:
                best_wer=val_wer
                S1_BEST_CKPT=STAGE1_OUT/'stage1_ft_best.pt'
                torch.save({'epoch':epoch,'model_state_dict':s1_model.state_dict(),
                            'vocab':idx2tok,'wer':val_wer},S1_BEST_CKPT)
                print(f'  → new best WER={best_wer:.4f}')

    # Training curve
    fig,ax=plt.subplots(1,2,figsize=(12,4))
    ax[0].plot(s1_train_losses); ax[0].set_title('CTC Loss'); ax[0].set_xlabel('step')
    ax[1].plot(range(10,CFG['s1_epochs']+1,10)[:len(s1_val_wers)],s1_val_wers)
    ax[1].set_title('Val WER'); ax[1].set_xlabel('epoch')
    plt.tight_layout(); plt.savefig(STAGE1_OUT/'stage1_ft_curves.png',dpi=120)
    plt.show()
    print(f'Stage 1 fine-tune complete. Best WER: {best_wer:.4f}')
    print(f'Checkpoint: {S1_BEST_CKPT}')

## 10. E2E Inference — Stage1 → Stage2 → HiFi-GAN → WAV

1. Load best Stage 2 checkpoint.
2. Load best Stage 1 model → predicted text + confidence.
3. Stage 2 DDPM sampling with confidence-weighted guidance.
4. HiFi-GAN vocoder → 16 kHz WAV.

In [ ]:
# ── Load best Stage 2 ─────────────────────────────────────────────────────
s2_ckpt = OUTPUT_DIR/'stage2_ft_best.pkl'
if not s2_ckpt.exists():
    # fallback: use latest checkpoint
    ckpts = sorted(OUTPUT_DIR.glob('stage2_ft_*.pkl'))
    s2_ckpt = ckpts[-1] if ckpts else None

if s2_ckpt and s2_ckpt.exists():
    payload = torch.load(s2_ckpt, map_location='cpu')
    sd = {k.replace('module.','',1):v for k,v in payload['model_state_dict'].items()}
    net.load_state_dict(sd)
    print(f'Loaded Stage 2 ckpt: {s2_ckpt} (step={payload.get("step","?")})')
else:
    print('No fine-tuned Stage 2 found, using LRS2 pretrained.')

net.eval()
# Recompute diff_hp with fast schedule (6 steps for inference speed)
# For full quality use T=400; for quick demo use fast schedule
FAST_BETA = [0.0001, 0.001, 0.01, 0.05, 0.2, 0.5]  # 6-step fast inference
USE_FAST  = False  # set True for quick demo, False for full 400-step quality

if USE_FAST:
    from Pipeline.third_party.LipVoicer.utils import diffwave_fast_inference_schedule
    inf_hp = diffwave_fast_inference_schedule(
        CFG['T'], CFG['beta_0'], CFG['beta_T'], beta=FAST_BETA
    )
else:
    inf_hp = diff_hp  # full 400-step

print(f'Inference steps: {inf_hp["T"]}')

In [ ]:
# ── Stage 1 inference → confidence ────────────────────────────────────────
if S1_BEST_CKPT and Path(S1_BEST_CKPT).exists():
    ckpt_s1 = torch.load(S1_BEST_CKPT, map_location='cpu')
    s1_model.load_state_dict(ckpt_s1['model_state_dict'])
    print(f'Loaded Stage 1 fine-tuned (WER={ckpt_s1["wer"]:.4f})')
else:
    print('Using Stage 1 as-is (random init or pretrained AVSR handles Stage 2 handoff).')

s1_model.eval()

@torch.no_grad()
def s1_infer(roi_np):
    """Returns (pred_text, seq_confidence) for a single clip."""
    roi_f = roi_np.astype(np.float32)/255.0
    roi_t = torch.from_numpy(roi_f).unsqueeze(0).unsqueeze(0).to(DEVICE) # (1,1,T,96,96)
    lens  = torch.tensor([roi_f.shape[0]], device=DEVICE)
    logits, feats = s1_model(roi_t, lens, return_feats=True)  # (1,T,V)
    probs = torch.softmax(logits, dim=-1)                      # (1,T,V)
    pred_ids = logits.argmax(-1)[0].cpu().tolist()
    pred_text = decode(pred_ids)
    # sequence-level confidence = mean max-prob over non-blank frames
    max_p = probs[0].max(-1).values  # (T,)
    seq_conf = float(max_p.mean())
    return pred_text, seq_conf, feats

# Run on test set
s1_results = []
for _,r in tqdm(test_df.iterrows(), total=len(test_df), desc='S1 test'):
    roi = np.load(ROI_DIR/r['speaker_id']/f"{r['clip_id']}.npz")['mouth_rois']
    pred,conf,_ = s1_infer(roi)
    s1_results.append({'clip_id':r['clip_id'],'gt':r['text'].lower().strip(),
                       'pred':pred,'conf':conf})
s1_df = pd.DataFrame(s1_results)
s1_wer = jiwer.wer(s1_df['gt'].tolist(), s1_df['pred'].tolist())
s1_cer = jiwer.cer(s1_df['gt'].tolist(), s1_df['pred'].tolist())
print(f'Stage 1 test WER={s1_wer:.4f}  CER={s1_cer:.4f}')
s1_df.to_csv(STAGE1_OUT/'test_predictions_s1.csv', index=False)

In [ ]:
# ── HiFi-GAN vocoder ──────────────────────────────────────────────────────
from Pipeline.third_party.LipVoicer.hifi_gan.generator import Generator
from Pipeline.third_party.LipVoicer.hifi_gan.env import AttrDict

# Standard 16kHz config (upsample product = 160 = hop_length)
HIFI_CFG = AttrDict({
    'resblock': '1',
    'upsample_rates': [8, 5, 4],          # 8×5×4 = 160 = hop_length ✓
    'upsample_kernel_sizes': [16, 10, 8],
    'upsample_initial_channel': 128,
    'resblock_kernel_sizes': [3, 7, 11],
    'resblock_dilation_sizes': [[1,3,5],[1,3,5],[1,3,5]],
})

vocoder = Generator(HIFI_CFG).to(DEVICE)

voc_payload = torch.load(HIFIGAN_CKPT, map_location='cpu')
# HiFi-GAN checkpoints store generator state under 'generator'
gen_sd = voc_payload.get('generator', voc_payload)
gen_sd = {k.replace('module.','',1):v for k,v in gen_sd.items()}
vocoder.load_state_dict(gen_sd)
vocoder.eval()
vocoder.remove_weight_norm()
print('HiFi-GAN loaded')

@torch.no_grad()
def mel_to_wav(mel_norm):
    """mel_norm: (80, L) normalized mel → wav tensor (T,)"""
    mel_denorm = denormalise_mel(mel_norm.unsqueeze(0).to(DEVICE))  # (1,80,L)
    wav = vocoder(mel_denorm).squeeze()                              # (T,)
    return wav.cpu()

In [ ]:
# ── Stage 2 sampling ─────────────────────────────────────────────────────
@torch.no_grad()
def sample_mel(roi_t, face_t, w_video=CFG['w_video']):
    """roi_t: (1,1,T,88,88)  face_t: (1,3,224,224) → mel (1,80,L)"""
    T_steps = inf_hp['T']
    Alpha   = inf_hp['Alpha']
    Alpha_bar = inf_hp['Alpha_bar']
    Sigma   = inf_hp['Sigma']

    mel_len = roi_t.shape[2] * int(CFG['vid_2_aud'])  # video frames → mel frames
    x = torch.randn(1, CFG['n_mels'], mel_len, device=DEVICE)

    for t in range(T_steps-1, -1, -1):
        ts = torch.full((1,1), t, device=DEVICE, dtype=torch.float32)
        with torch.autocast('cuda', dtype=torch.bfloat16):
            eps_cond   = net(x, roi_t, face_t, ts, cond_drop_prob=0)
            eps_uncond = net(x, roi_t, face_t, ts, cond_drop_prob=1)
        eps = (1+w_video)*eps_cond - w_video*eps_uncond  # classifier-free guidance
        eps = eps.float()
        x = (x - (1-Alpha[t])/(1-Alpha_bar[t]).sqrt() * eps) / Alpha[t].sqrt()
        if t > 0:
            x = x + Sigma[t] * torch.randn_like(x)
    return x  # (1,80,L)

# ── Generate samples for first N test clips ────────────────────────────────
N_DEMO = min(10, len(test_df))  # generate 10 samples
audio_dir = OUTPUT_DIR / 'audio_samples'
audio_dir.mkdir(exist_ok=True)

gen_records = []
for i, (_,row) in enumerate(tqdm(test_df.head(N_DEMO).iterrows(),
                                  total=N_DEMO, desc='E2E generate')):
    cid = row['clip_id']; spk = row['speaker_id']

    # Load ROI full sequence (no windowing for inference)
    roi_np = np.load(ROI_DIR/spk/f'{cid}.npz')['mouth_rois']  # (T,96,96)
    roi_t  = torch.FloatTensor(
        roi_tfm('val')(roi_np)   # center-crop → (T,88,88)
    ).unsqueeze(0).unsqueeze(0).to(DEVICE)                     # (1,1,T,88,88)

    face_t = face_tfm()(
        Image.open(FACE_DIR/f'{cid}_face.jpg').convert('RGB')
    ).unsqueeze(0).to(DEVICE)                                   # (1,3,224,224)

    mel_gen = sample_mel(roi_t, face_t)                         # (1,80,L)
    wav     = mel_to_wav(mel_gen.squeeze(0))                    # (T_wav,)

    # Save WAV
    wav_path = audio_dir / f'{cid}_generated.wav'
    torchaudio.save(str(wav_path), wav.unsqueeze(0), CFG['sr'])

    # Also save ground-truth for comparison
    gt_mel = normalise_mel(torch.load(MEL_DIR/f'{cid}.wav.spec'))
    gt_wav = mel_to_wav(gt_mel)
    gt_wav_path = audio_dir / f'{cid}_gt.wav'
    torchaudio.save(str(gt_wav_path), gt_wav.unsqueeze(0), CFG['sr'])

    gen_records.append({'clip_id':cid,'generated':str(wav_path),'gt':str(gt_wav_path)})
    print(f'  [{i+1}/{N_DEMO}] {cid}: {wav.shape[0]/CFG["sr"]:.2f}s generated')

gen_df = pd.DataFrame(gen_records)
print(f'\nGenerated {N_DEMO} samples → {audio_dir}')

In [ ]:
# ── Mel-spec comparison plots ──────────────────────────────────────────────
fig, axes = plt.subplots(2, min(4,N_DEMO), figsize=(16, 6))
for i, rec in enumerate(gen_records[:min(4,N_DEMO)]):
    cid = rec['clip_id']
    gt_mel   = torch.load(MEL_DIR/f'{cid}.wav.spec').numpy()
    # Load generated mel by re-computing from saved WAV
    gen_wav,_ = torchaudio.load(rec['generated'])
    gen_mel   = stft_fn.mel_spectrogram(
                    gen_wav.to(DEVICE)/gen_wav.abs().max().clamp(1e-8)
                ).squeeze(0).cpu().numpy()
    L = min(gt_mel.shape[1], gen_mel.shape[1], 200)
    axes[0,i].imshow(gt_mel[:,:L],aspect='auto',origin='lower')
    axes[0,i].set_title(f'GT: {cid[:15]}')
    axes[1,i].imshow(gen_mel[:,:L],aspect='auto',origin='lower')
    axes[1,i].set_title('Generated')
for ax in axes.flat: ax.axis('off')
plt.suptitle('Mel-spectrogram: Ground Truth vs Generated')
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'mel_comparison.png',dpi=120)
plt.show()

## 11. Metrics — STOI, PESQ, WER

Evaluate generated audio quality on test set clips.

In [ ]:
from pystoi import stoi
try:
    from pesq import pesq
except ImportError:
    pesq = None

def compute_metrics(gt_wav_path, gen_wav_path, sr=16000):
    gt,  gs  = torchaudio.load(gt_wav_path)
    gen, gns = torchaudio.load(gen_wav_path)
    if gs  != sr: gt  = torchaudio.functional.resample(gt,  gs,  sr)
    if gns != sr: gen = torchaudio.functional.resample(gen, gns, sr)
    gt  = gt.mean(0).numpy()
    gen = gen.mean(0).numpy()
    # Align lengths
    L = min(len(gt), len(gen))
    gt, gen = gt[:L], gen[:L]
    if L < sr//4: return None  # too short
    try:
        stoi_score = stoi(gt, gen, sr, extended=False)
        pesq_score = pesq(sr, gt, gen, 'wb') if pesq else None
    except Exception:
        return None
    return {
        'stoi': float(stoi_score),
        'pesq': float(pesq_score) if pesq_score is not None else None,
    }

metric_rows = []
for rec in tqdm(gen_records, desc='metrics'):
    m = compute_metrics(rec['gt'], rec['generated'])
    if m:
        m['clip_id'] = rec['clip_id']
        metric_rows.append(m)

if metric_rows:
    met_df = pd.DataFrame(metric_rows)
    print(f'\nSTOI: mean={met_df["stoi"].mean():.4f}  std={met_df["stoi"].std():.4f}')
    print(f'PESQ: mean={met_df["pesq"].mean():.4f}  std={met_df["pesq"].std():.4f}')
    display(met_df)
    met_df.to_csv(OUTPUT_DIR/'test_metrics.csv', index=False)
else:
    print('No metrics computed (pystoi/pesq may need install or audio too short)')

In [ ]:
# ── Full test-set evaluation (all 108 clips) ───────────────────────────────
print('Full test-set E2E evaluation...')
full_records = []
for _,row in tqdm(test_df.iterrows(), total=len(test_df), desc='test E2E'):
    cid = row['clip_id']; spk = row['speaker_id']
    roi_np = np.load(ROI_DIR/spk/f'{cid}.npz')['mouth_rois']
    roi_t  = torch.FloatTensor(roi_tfm('val')(roi_np)).unsqueeze(0).unsqueeze(0).to(DEVICE)
    face_t = face_tfm()(Image.open(FACE_DIR/f'{cid}_face.jpg').convert('RGB')
                        ).unsqueeze(0).to(DEVICE)

    mel_g  = sample_mel(roi_t, face_t).squeeze(0)  # (80,L)
    wav_g  = mel_to_wav(mel_g)
    wav_p  = audio_dir/f'{cid}_gen_full.wav'
    torchaudio.save(str(wav_p), wav_g.unsqueeze(0), CFG['sr'])

    gt_mel = normalise_mel(torch.load(MEL_DIR/f'{cid}.wav.spec'))
    gt_wav = mel_to_wav(gt_mel)
    gt_p   = audio_dir/f'{cid}_gt_full.wav'
    torchaudio.save(str(gt_p), gt_wav.unsqueeze(0), CFG['sr'])

    m = compute_metrics(str(gt_p), str(wav_p))
    if m: m.update({'clip_id':cid,'gt_text':row['text'].lower().strip()})
    else: m = {'clip_id':cid,'stoi':None,'pesq':None,'gt_text':row['text']}
    full_records.append(m)

full_df = pd.DataFrame(full_records)
valid   = full_df[full_df['stoi'].notna()]
print(f'\nTest set ({len(valid)}/{len(full_df)} valid):')
print(f'  STOI: {valid["stoi"].mean():.4f} ± {valid["stoi"].std():.4f}')
print(f'  PESQ: {valid["pesq"].mean():.4f} ± {valid["pesq"].std():.4f}')
full_df.to_csv(OUTPUT_DIR/'test_full_metrics.csv', index=False)

## 12. Ablation — Guidance Weights + Failure Analysis

In [ ]:
# ── Guidance weight ablation on 20 val clips ──────────────────────────────
ABLATION_CLIPS = val_df.head(20)
W_VIDEO_OPTIONS = [0.0, 1.0, 2.0, 4.0]

ablation_rows = []
for w_v in W_VIDEO_OPTIONS:
    clip_metrics = []
    for _,row in tqdm(ABLATION_CLIPS.iterrows(), total=len(ABLATION_CLIPS),
                      desc=f'w_video={w_v}', leave=False):
        cid = row['clip_id']; spk = row['speaker_id']
        roi_np = np.load(ROI_DIR/spk/f'{cid}.npz')['mouth_rois']
        roi_t  = torch.FloatTensor(roi_tfm('val')(roi_np)).unsqueeze(0).unsqueeze(0).to(DEVICE)
        face_t = face_tfm()(Image.open(FACE_DIR/f'{cid}_face.jpg').convert('RGB')
                            ).unsqueeze(0).to(DEVICE)

        mel_g  = sample_mel(roi_t, face_t, w_video=w_v).squeeze(0)
        wav_g  = mel_to_wav(mel_g)
        gt_mel = normalise_mel(torch.load(MEL_DIR/f'{cid}.wav.spec'))
        gt_wav = mel_to_wav(gt_mel)

        wp = audio_dir/f'abl_{cid}_w{w_v}.wav'
        gp = audio_dir/f'abl_{cid}_gt.wav'
        torchaudio.save(str(wp), wav_g.unsqueeze(0), CFG['sr'])
        torchaudio.save(str(gp), gt_wav.unsqueeze(0), CFG['sr'])
        m = compute_metrics(str(gp), str(wp))
        if m: clip_metrics.append(m)

    if clip_metrics:
        cm = pd.DataFrame(clip_metrics)
        ablation_rows.append({
            'w_video': w_v,
            'stoi_mean': cm['stoi'].mean(), 'stoi_std': cm['stoi'].std(),
            'pesq_mean': cm['pesq'].mean(), 'pesq_std': cm['pesq'].std(),
        })

abl_df = pd.DataFrame(ablation_rows)
print('\nGuidance weight ablation:')
display(abl_df.round(4))
abl_df.to_csv(OUTPUT_DIR/'ablation_guidance.csv', index=False)

# Plot
fig,ax=plt.subplots(1,2,figsize=(10,4))
ax[0].bar(abl_df['w_video'].astype(str), abl_df['stoi_mean'],
          yerr=abl_df['stoi_std'], capsize=5)
ax[0].set_xlabel('w_video'); ax[0].set_ylabel('STOI'); ax[0].set_title('STOI vs guidance')
ax[1].bar(abl_df['w_video'].astype(str), abl_df['pesq_mean'],
          yerr=abl_df['pesq_std'], capsize=5)
ax[1].set_xlabel('w_video'); ax[1].set_ylabel('PESQ'); ax[1].set_title('PESQ vs guidance')
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'ablation_guidance.png', dpi=120)
plt.show()

In [ ]:
# ── Confidence-weighted guidance ──────────────────────────────────────────
# Stage 1 confidence c ∈ [0,1] scales w_video: effective_w = w_video * c
# Low confidence → weaker text guidance (avoids hallucination on ambiguous lips)

conf_rows = []
for _,row in tqdm(ABLATION_CLIPS.iterrows(), total=len(ABLATION_CLIPS),
                  desc='conf-weighted', leave=False):
    cid = row['clip_id']; spk = row['speaker_id']
    roi_np = np.load(ROI_DIR/spk/f'{cid}.npz')['mouth_rois']

    _,conf,_ = s1_infer(roi_np)
    w_eff    = CFG['w_video'] * float(conf)  # confidence-scaled guidance

    roi_t  = torch.FloatTensor(roi_tfm('val')(roi_np)).unsqueeze(0).unsqueeze(0).to(DEVICE)
    face_t = face_tfm()(Image.open(FACE_DIR/f'{cid}_face.jpg').convert('RGB')
                        ).unsqueeze(0).to(DEVICE)
    mel_g  = sample_mel(roi_t, face_t, w_video=w_eff).squeeze(0)
    wav_g  = mel_to_wav(mel_g)
    gt_mel = normalise_mel(torch.load(MEL_DIR/f'{cid}.wav.spec'))
    gt_wav = mel_to_wav(gt_mel)

    wp = audio_dir/f'conf_{cid}.wav'
    gp = audio_dir/f'conf_{cid}_gt.wav'
    torchaudio.save(str(wp),wav_g.unsqueeze(0),CFG['sr'])
    torchaudio.save(str(gp),gt_wav.unsqueeze(0),CFG['sr'])
    m=compute_metrics(str(gp),str(wp))
    if m: m.update({'clip_id':cid,'conf':conf,'w_eff':w_eff}); conf_rows.append(m)

if conf_rows:
    conf_df=pd.DataFrame(conf_rows)
    print(f'Confidence-weighted guidance:')
    print(f'  STOI: {conf_df["stoi"].mean():.4f}  PESQ: {conf_df["pesq"].mean():.4f}')
    # Compare with fixed w_video=2
    fixed = abl_df[abl_df['w_video']==2.0]
    if len(fixed):
        print(f'  Fixed w=2.0:   STOI={fixed.iloc[0]["stoi_mean"]:.4f}')
        print(f'  Conf-weighted: STOI={conf_df["stoi"].mean():.4f}')
    conf_df.to_csv(OUTPUT_DIR/'ablation_conf_weighted.csv', index=False)

In [ ]:
# ── Failure analysis: bucket by clip length and S1 confidence ─────────────
if metric_rows or conf_rows:
    src = conf_df if conf_rows else met_df
    if 'conf' not in src.columns:
        confs={r['clip_id']:s1_infer(np.load(ROI_DIR/
               test_df[test_df.clip_id==r['clip_id']].iloc[0]['speaker_id']/
               f"{r['clip_id']}.npz")['mouth_rois'])[1]
               for r in metric_rows}
        src['conf']=src['clip_id'].map(confs)

    src['conf_bucket'] = pd.cut(src['conf'],bins=[0,.3,.6,.8,1.0],
                                labels=['very_low','low','mid','high'],include_lowest=True)
    buckets = src.groupby('conf_bucket',observed=False)[['stoi','pesq']].mean().round(4)
    print('\nPerformance by S1 confidence bucket:')
    display(buckets)
    buckets.to_csv(OUTPUT_DIR/'failure_buckets.csv')

# ── Summary report ─────────────────────────────────────────────────────────
def clean_metric(value):
    if value is None or pd.isna(value):
        return None
    return float(value)

report = {
    'stage1': {
        'pretrained_wer': float(S1_PRETRAINED_WER),
        'finetuned_wer': float(best_wer) if NEED_S1_FINETUNE else None,
        'test_wer': float(s1_wer), 'test_cer': float(s1_cer),
    },
    'stage2': {
        'best_val_loss': clean_metric(best_val_loss),
        'test_stoi': clean_metric(valid['stoi'].mean()) if len(valid) else None,
        'test_pesq': clean_metric(valid['pesq'].mean()) if len(valid) else None,
    },
    'best_guidance': (
        abl_df.loc[abl_df['stoi_mean'].idxmax(),'w_video']
        if len(abl_df) else None
    ),
}
(OUTPUT_DIR/'final_report.json').write_text(json.dumps(report, indent=2, default=str))
print('\n=== FINAL REPORT ===')
print(json.dumps(report, indent=2, default=str))